In [3]:
# !pip3 install spacy 
# !pip3 install scikit-learn
!python3 -m spacy download en_core_web_sm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 60.1 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 24.3.1 -> 25.0
[notice] To update, run: pip3 install --upgrade pip
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [4]:
import spacy

nlp = spacy.load("en_core_web_sm")  # Load a spaCy model (you might need a larger model for better accuracy)

def classify_query(query, categories):
    doc = nlp(query)
    for cat, keywords in categories.items():
        for token in doc:
            if token.lemma_ in keywords:  # Check lemmas for better matching
                return cat
    return "Unclassified"  # Default if no category matches

# Example usage:
categories = {
    "Auto": ["car", "vehicle", "loan", "insurance", "drive", "repair"],
    "Agriculture": ["farm", "crop", "tractor", "harvest", "soil", "seed"],
    "Finance": ["money", "bank", "investment", "loan", "interest"] #Note: "loan" appears in multiple categories. Requires more sophisticated approach for such overlaps.
}

query = "I need a car loan."
classification = classify_query(query, categories)
print(f"Query: {query}\nClassification: {classification}")

query = "My tractor needs repair."
classification = classify_query(query, categories)
print(f"Query: {query}\nClassification: {classification}")

query = "What are the investment options?"
classification = classify_query(query, categories)
print(f"Query: {query}\nClassification: {classification}")

Query: I need a car loan.
Classification: Auto
Query: My tractor needs repair.
Classification: Auto
Query: What are the investment options?
Classification: Finance


In [11]:
import csv

def load_csv_to_tuples(filepath):
    try:
        with open(filepath, 'r', encoding='utf-8') as csvfile:
            reader = csv.reader(csvfile)
            next(reader, None)  
            data = [(row[0].split(",")[0], row[0].split(",")[1]) for row in reader]
        return data
    except FileNotFoundError:
        print(f"Error: File not found at {filepath}")
        return []
    except Exception as e:
        print(f"An error occurred: {e}")
        return []

In [2]:
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
# ... (Your data loading and preprocessing code) ...

nlp = spacy.load("en_core_web_sm")

#Feature Extraction function
def extract_features(text):
    doc = nlp(text)
    features = [token.lemma_ for token in doc]  #Using lemmas as features for simplicity
    return " ".join(features)

data_files = ["agriculture_query_data.csv", "agriculture_query_data.csv", "truck_query_data.csv"]
data = []
for data_file in data_files:
    data.extend(load_csv_to_tuples(data_file))

texts, labels = zip(*data)
X = [extract_features(text) for text in texts] #Apply feature extraction function
y = labels

#Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

vectorizer = TfidfVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

model = LogisticRegression()
model.fit(X_train_vec, y_train)

#Predict on the test set
y_pred = model.predict(X_test_vec)

#Evaluate the model (optional)
#from sklearn.metrics import accuracy_score
#accuracy = accuracy_score(y_test, y_pred)
#print(f"Accuracy: {accuracy}")


In [4]:
#Classify a new query
queries = ["What is a level 1 automotive manufacturer?", "I'm looking for farm equipment.", "My tractor needs repair."]
for new_query in queries:
    # new_query = "I'm looking for farm equipment."
    new_query_vec = vectorizer.transform([extract_features(new_query)])
    prediction = model.predict(new_query_vec)[0]
    print(f"Query: {new_query}\nClassification: {prediction}")

Query: What is a level 1 automotive manufacturer?
Classification: Auto
Query: I'm looking for farm equipment.
Classification: Auto
Query: My tractor needs repair.
Classification: Auto


In [13]:
data_files = ["agriculture_query_data.csv", "agriculture_query_data.csv", "truck_query_data.csv"]
print(data_files[0])
data = load_csv_to_tuples(data_files[0])

agriculture_query_data.csv
An error occurred: list index out of range
